# Phase 5 - Modeling

Four predictors: mean baseline, day-of-week baseline, Ridge regression, Random Forest. Chronological train/test split at 2023-08-01. Then TimeSeriesSplit CV, diagnostic figures, and per-category RF models.

In [1]:
import sys
sys.path.append("../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from features import build_features
from viz import save_fig

sns.set_style("whitegrid")
sns.set_palette("colorblind")
plt.rcParams["figure.dpi"] = 120

In [2]:
merged = pd.read_csv("../data/processed/merged.csv", parse_dates=["date"])
df = build_features(merged)

# drop the first 7 rows - rolling/lag features have NaN there
df = df.dropna(subset=["crime_7day_avg", "tmax_f_lag1", "prcp_mm_lag1"]).reset_index(drop=True)
print("after dropna:", df.shape)

after dropna: (2550, 33)


## Train/test split

Chronological: train ends 2023-07-31, test begins 2023-08-01. About 17 months of test data.

In [3]:
train = df[df["date"] < "2023-08-01"]
test = df[df["date"] >= "2023-08-01"]

feature_cols = ["tmax_f", "tmin_f", "prcp_mm", "awnd_ms",
                "day_of_week", "month", "is_weekend", "is_holiday",
                "is_rainy", "is_hot", "is_cold",
                "tmax_f_lag1", "prcp_mm_lag1", "crime_7day_avg",
                "is_covid_year", "is_nye", "is_july4"]

X_train = train[feature_cols]
y_train = train["total_crimes"]
X_test = test[feature_cols]
y_test = test["total_crimes"]

print(f"train: {len(train)} rows, test: {len(test)} rows")
print(f"train range: {train['date'].min().date()} -> {train['date'].max().date()}")
print(f"test  range: {test['date'].min().date()} -> {test['date'].max().date()}")

train: 2031 rows, test: 519 rows
train range: 2018-01-08 -> 2023-07-31
test  range: 2023-08-01 -> 2024-12-31


## Baselines

Two of them. The mean baseline is the bare minimum. The day-of-week baseline is what the rubric calls a "strong" baseline - I have to beat it to claim weather actually adds predictive value.

In [4]:
# baseline 1 - just predict the train mean
mean_pred = np.full(len(y_test), y_train.mean())
print(f"mean baseline MAE: {mean_absolute_error(y_test, mean_pred):.2f}")
print(f"mean baseline R2:  {r2_score(y_test, mean_pred):.3f}")

mean baseline MAE: 74.95
mean baseline R2:  -0.589


In [5]:
# baseline 2 - day-of-week mean from train
dow_means = train.groupby("day_of_week")["total_crimes"].mean()
dow_pred = test["day_of_week"].map(dow_means).values
print(f"day-of-week baseline MAE: {mean_absolute_error(y_test, dow_pred):.2f}")
print(f"day-of-week baseline R2:  {r2_score(y_test, dow_pred):.3f}")

day-of-week baseline MAE: 75.19
day-of-week baseline R2:  -0.571


## Ridge regression

Standardize the features first - Ridge is sensitive to scale.

In [6]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_train)
ridge_pred = ridge.predict(X_test_s)

print(f"Ridge MAE:  {mean_absolute_error(y_test, ridge_pred):.2f}")
print(f"Ridge RMSE: {np.sqrt(mean_squared_error(y_test, ridge_pred)):.2f}")
print(f"Ridge R2:   {r2_score(y_test, ridge_pred):.3f}")

Ridge MAE:  40.49
Ridge RMSE: 54.87
Ridge R2:   0.456


## Random Forest

In [7]:
rf = RandomForestRegressor(n_estimators=200, max_depth=15,
                           min_samples_leaf=5, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print(f"RF MAE:  {mean_absolute_error(y_test, rf_pred):.2f}")
print(f"RF RMSE: {np.sqrt(mean_squared_error(y_test, rf_pred)):.2f}")
print(f"RF R2:   {r2_score(y_test, rf_pred):.3f}")

RF MAE:  39.37
RF RMSE: 54.50
RF R2:   0.463


## Comparison table

In [8]:
def metrics(y, p):
    return (mean_absolute_error(y, p),
            np.sqrt(mean_squared_error(y, p)),
            r2_score(y, p))

rows = []
for name, pred in [("mean_baseline", mean_pred),
                   ("dow_baseline", dow_pred),
                   ("ridge", ridge_pred),
                   ("random_forest", rf_pred)]:
    mae, rmse, r2 = metrics(y_test, pred)
    rows.append({"model": name, "test_mae": round(mae, 2),
                 "test_rmse": round(rmse, 2), "test_r2": round(r2, 3)})
compare = pd.DataFrame(rows)
compare

,model,test_mae,test_rmse,test_r2
0,mean_baseline,74.95,93.75,-0.589
1,dow_baseline,75.19,93.24,-0.571
2,ridge,40.49,54.87,0.456
3,random_forest,39.37,54.50,0.463


## TimeSeriesSplit CV (5 folds) on Ridge and RF

Random KFold would leak future into past - TimeSeriesSplit respects time order. CV runs on the training data only; test stays untouched.

In [9]:
tscv = TimeSeriesSplit(n_splits=5)

ridge_cv = -cross_val_score(Ridge(alpha=1.0), X_train_s, y_train,
                            cv=tscv, scoring="neg_mean_absolute_error")
rf_cv = -cross_val_score(RandomForestRegressor(n_estimators=200, max_depth=15,
                                               min_samples_leaf=5, random_state=42, n_jobs=-1),
                         X_train, y_train,
                         cv=tscv, scoring="neg_mean_absolute_error")

print(f"Ridge CV MAE: {ridge_cv.mean():.2f} +/- {ridge_cv.std():.2f}")
print(f"RF CV MAE:    {rf_cv.mean():.2f} +/- {rf_cv.std():.2f}")

compare["cv_mae_mean"] = [np.nan, np.nan, round(ridge_cv.mean(), 2), round(rf_cv.mean(), 2)]
compare["cv_mae_std"] = [np.nan, np.nan, round(ridge_cv.std(), 2), round(rf_cv.std(), 2)]

import os
os.makedirs("../outputs", exist_ok=True)
compare.to_csv("../outputs/comparison.csv", index=False)
print("\nsaved ../outputs/comparison.csv")
compare

Ridge CV MAE: 62.79 +/- 30.75
RF CV MAE:    66.50 +/- 37.17

saved ../outputs/comparison.csv


,model,test_mae,test_rmse,test_r2,cv_mae_mean,cv_mae_std
0,mean_baseline,74.95,93.75,-0.589,NaN,NaN
1,dow_baseline,75.19,93.24,-0.571,NaN,NaN
2,ridge,40.49,54.87,0.456,62.79,30.75
3,random_forest,39.37,54.50,0.463,66.50,37.17


In [10]:
# paired t-test on per-fold CV MAE between Ridge and RF
from scipy import stats
t, p = stats.ttest_rel(ridge_cv, rf_cv)
print(f"paired t-test on per-fold CV MAE (Ridge vs RF):")
print(f"  t = {t:.3f}, p = {p:.4g}")
if p < 0.05:
    print("  RF is significantly better than Ridge at alpha=0.05")
else:
    print("  RF and Ridge are statistically indistinguishable")

paired t-test on per-fold CV MAE (Ridge vs RF):
  t = -0.769, p = 0.4847
  RF and Ridge are statistically indistinguishable


## Diagnostic figures

In [11]:
# fig 10 - actual vs predicted (RF)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, rf_pred, alpha=0.5)
lims = [min(y_test.min(), rf_pred.min()) - 20, max(y_test.max(), rf_pred.max()) + 20]
ax.plot(lims, lims, color="red", linestyle="--", label="y = x")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("actual total crimes")
ax.set_ylabel("predicted total crimes (RF)")
ax.set_title("Random Forest - actual vs. predicted (test set)")
ax.legend()
save_fig("fig_10_actual_vs_predicted")

In [12]:
# fig 11 - residuals over time
resid = y_test.values - rf_pred
fig, ax = plt.subplots(figsize=(11, 5))
ax.scatter(test["date"], resid, alpha=0.5)
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("date")
ax.set_ylabel("residual (actual - predicted)")
ax.set_title("RF residuals over time, 2023-08 -> 2024-12")
save_fig("fig_11_residuals_time")

In [13]:
# fig 12 - residual histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(resid, bins=30, edgecolor="black")
ax.axvline(0, color="red", linestyle="--")
ax.set_xlabel("residual (actual - predicted)")
ax.set_ylabel("count")
ax.set_title("RF residual distribution (test set)")
save_fig("fig_12_residual_hist")

In [14]:
# fig 13 - top 15 feature importance
imp = pd.DataFrame({"feature": feature_cols, "importance": rf.feature_importances_})
imp = imp.sort_values("importance", ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp["feature"], imp["importance"])
ax.set_xlabel("importance")
ax.set_title("Random Forest feature importance (top 15)")
save_fig("fig_13_feature_importance")
imp.iloc[::-1]

,feature,importance
13,crime_7day_avg,0.791260
0,tmax_f,0.050321
3,awnd_ms,0.025599
1,tmin_f,0.024650
11,tmax_f_lag1,0.022814
4,day_of_week,0.022422
2,prcp_mm,0.016783
5,month,0.016157
12,prcp_mm_lag1,0.011523
7,is_holiday,0.007500


### Fig 14 - Permutation importance

`feature_importances_` is split-count based and tends to over-weight the feature that the tree splits on most. Permutation importance shuffles each column on the test set and measures the drop in test R² — a more honest measure of what the model genuinely relies on at inference. I use 10 repeats so the mean is stable.

In [15]:
# fig 14 - permutation importance on the test set
from sklearn.inspection import permutation_importance
result = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
perm = pd.DataFrame({"feature": feature_cols, "importance": result.importances_mean})
perm = perm.sort_values("importance", ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(perm["feature"], perm["importance"])
ax.set_xlabel("permutation importance (mean drop in test R²)")
ax.set_title("RF permutation importance on test set (top 15)")
save_fig("fig_14_permutation_importance")
perm.iloc[::-1]

,feature,importance
13,crime_7day_avg,6.217489e-01
0,tmax_f,1.141722e-01
4,day_of_week,3.988840e-02
5,month,1.395167e-02
2,prcp_mm,1.284909e-02
3,awnd_ms,1.248642e-02
1,tmin_f,8.374706e-03
12,prcp_mm_lag1,3.734610e-03
6,is_weekend,7.243025e-04
8,is_rainy,2.163407e-04


**Comparison with fig 13:** Both rankings put `crime_7day_avg` at the top — the model genuinely leans on the recent rolling average. After that, the rankings diverge. Built-in importance is split-count based and inflates `crime_7day_avg` to ~79% with everything else compressed below 5%. Permutation importance is more honest: it measures how much test R² actually drops when each feature is shuffled, and gives weather features (especially `tmax_f`, `month`, `day_of_week`) much clearer relative weight. The signal in this model isn't *only* the rolling average; weather and calendar features each carry real predictive value when the rolling average is held intact.

## Per-category models

Train a separate RF on each crime category. The hypothesis from the EDA: weather-sensitive categories (assault, battery) should fit better than weather-independent ones (burglary, narcotics).

In [16]:
cats = ["total_crimes", "theft", "battery", "burglary", "assault", "narcotics", "criminal_damage"]
rows = []
for c in cats:
    yt = train[c]
    ye = test[c]
    m = RandomForestRegressor(n_estimators=200, max_depth=15,
                              min_samples_leaf=5, random_state=42, n_jobs=-1)
    m.fit(X_train, yt)
    pred = m.predict(X_test)
    rows.append({
        "category": c,
        "r2": round(r2_score(ye, pred), 3),
        "mae": round(mean_absolute_error(ye, pred), 2),
        "rmse": round(np.sqrt(mean_squared_error(ye, pred)), 2),
    })
per_cat = pd.DataFrame(rows)
per_cat.to_csv("../outputs/per_category.csv", index=False)
print("saved ../outputs/per_category.csv")
per_cat

saved ../outputs/per_category.csv


,category,r2,mae,rmse
0,total_crimes,0.463,39.37,54.50
1,theft,0.269,17.00,21.76
2,battery,0.403,13.13,16.95
3,burglary,-0.435,5.89,7.43
4,assault,0.090,8.48,10.68
5,narcotics,-5.669,12.07,14.10
6,criminal_damage,0.236,9.51,12.62


In [17]:
# fig 15 - per-category R2 bar
fig, ax = plt.subplots(figsize=(9, 5))
ordered = per_cat.sort_values("r2", ascending=False)
sns.barplot(data=ordered, x="category", y="r2", ax=ax)
ax.set_xlabel("category")
ax.set_ylabel("test R²")
ax.set_title("Per-category Random Forest R² (test set)")
ax.axhline(0, color="black", linewidth=0.7)
plt.xticks(rotation=20)
save_fig("fig_15_per_category_r2")

## Save the main RF

Pickle the total-crime RF so the report notebook can load it without retraining.

In [18]:
with open("../outputs/rf_model.pkl", "wb") as f:
    pickle.dump(rf, f)
print("saved ../outputs/rf_model.pkl")

saved ../outputs/rf_model.pkl


Comparison table + 5-fold time-series CV done. Five diagnostic figures saved. Per-category R² saved. RF pickled. Phase 5 complete.